In [1]:
import torch
import torch.nn as nn

from src.quadratic_annealing_optimizer import QuadraticAnnealingOptimizer
from src.models import QuadraticMLP
from src.utils import build_sampler, data_load_and_prep
from src.training import train
from src.newton_optimizer import NewtonOptimizer

EXPERIMENT_NAME = "optimizer-comparison-mlp-iris-q"

## Idea

For the current parameter vector $w$, the batch loss is approximated locally by a second-order Taylor expansion:

$$
\mathcal{L}(w + \Delta) \approx \mathcal{L}(w) + g^\top \Delta + \frac{1}{2}\Delta^\top H\Delta,
$$

where $g = \nabla \mathcal{L}(w)$ and $H = \nabla^2 \mathcal{L}(w)$.

The annealer does not optimize the full continuous update directly. Instead, for a selected subset of parameters, it chooses binary variables $z_i \in \{0,1\}$ that encode the sign of a fixed-size step:

$$
\Delta_i = \eta(2z_i - 1),
$$

so $z_i = 1$ means a $+\eta$ step and $z_i = 0$ means a $-\eta$ step. Substituting this into the quadratic model turns the local optimization problem into a QUBO/BQM:

$$
E(z) = \sum_i a_i z_i + \sum_{i<j} b_{ij} z_i z_j + c.
$$

The annealer minimizes $E(z)$, then the proposed update is applied to the network parameters and accepted only if the true loss decreases.

## Loading sample Iris data for training

In [4]:
train_loader, test_loader = data_load_and_prep("iris", test_size=0.3, random_state=42, batch_size='full')
NUM_EPOCHS = 30

# Classical optimization for benchmarking

In [8]:
loss_fn = nn.CrossEntropyLoss()
adam_model = QuadraticMLP(4, [4], 3)
classical_device = "cpu" 
adam_optimizer = torch.optim.Adam(adam_model.parameters(), 
                             lr=0.01,
                             betas=[0.9, 0.999],
                             )

## Optimization using Adam optimizaer

In [4]:
train(
    model=adam_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=adam_optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS, 
    experiment_name = EXPERIMENT_NAME,
    run_name="adam-optimizer"
)

/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/01 12:08:48 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 000 | train_loss=1.2309 | train_acc=0.286 | test_loss=1.2130 | test_acc=0.289 | 
Epoch 005 | train_loss=1.1055 | train_acc=0.333 | test_loss=1.0997 | test_acc=0.378 | 
Epoch 010 | train_loss=0.9909 | train_acc=0.790 | test_loss=0.9968 | test_acc=0.800 | 
Epoch 015 | train_loss=0.8848 | train_acc=0.819 | test_loss=0.8999 | test_acc=0.800 | 
Epoch 020 | train_loss=0.7892 | train_acc=0.781 | test_loss=0.8105 | test_acc=0.756 | 
Epoch 025 | train_loss=0.7074 | train_acc=0.819 | test_loss=0.7324 | test_acc=0.756 | 
Epoch 029 | train_loss=0.6532 | train_acc=0.867 | test_loss=0.6797 | test_acc=0.800 | 


{'run_id': 'aa31b2d513984f4b92d2e1e272aa3d84',
 'experiment_id': '831960980176491418',
 'experiment_name': 'optimizer-comparison-mlp-iris-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'optimization_time_sec': 0.19850049999877228,
 'sum_epoch_optimization_time_sec': 0.19850049999877228,
 'train_evaluation_time_sec': 0.014550606992997928,
 'test_evaluation_time_sec': 0.008799848003036459,
 'evaluation_time_sec': 0.023350454996034387,
 'training_time_sec': 0.32570733900138293,
 'final_train_loss': 0.6531727313995361,
 'final_test_loss': 0.6796792149543762,
 'final_train_metric': 0.8666666666666667,
 'final_test_metric': 0.8,
 'optimizer': 'Adam',
 'seed': None,
 'epochs_completed': 30,
 'early_stopped': False,
 'early_stop_reason': None,
 'zero_acceptance_patience': 6,
 'final_zero_acceptance_streak': 0}

## Optimization using Adam optimizaer on cuda

In [ ]:
loss_fn = nn.CrossEntropyLoss()
adam_model = QuadraticMLP(4, [4], 3)
classical_device = "cuda" 
adam_optimizer = torch.optim.Adam(adam_model.parameters(), 
                             lr=0.01,
                             betas=[0.9, 0.999],
                             )

In [ ]:
train(
    model=adam_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=adam_optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS, 
    experiment_name = EXPERIMENT_NAME,
    run_name="adam-optimizer-cuda"
)

/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/01 12:08:48 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 000 | train_loss=1.2309 | train_acc=0.286 | test_loss=1.2130 | test_acc=0.289 | 
Epoch 005 | train_loss=1.1055 | train_acc=0.333 | test_loss=1.0997 | test_acc=0.378 | 
Epoch 010 | train_loss=0.9909 | train_acc=0.790 | test_loss=0.9968 | test_acc=0.800 | 
Epoch 015 | train_loss=0.8848 | train_acc=0.819 | test_loss=0.8999 | test_acc=0.800 | 
Epoch 020 | train_loss=0.7892 | train_acc=0.781 | test_loss=0.8105 | test_acc=0.756 | 
Epoch 025 | train_loss=0.7074 | train_acc=0.819 | test_loss=0.7324 | test_acc=0.756 | 
Epoch 029 | train_loss=0.6532 | train_acc=0.867 | test_loss=0.6797 | test_acc=0.800 | 


{'run_id': 'aa31b2d513984f4b92d2e1e272aa3d84',
 'experiment_id': '831960980176491418',
 'experiment_name': 'optimizer-comparison-mlp-iris-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'optimization_time_sec': 0.19850049999877228,
 'sum_epoch_optimization_time_sec': 0.19850049999877228,
 'train_evaluation_time_sec': 0.014550606992997928,
 'test_evaluation_time_sec': 0.008799848003036459,
 'evaluation_time_sec': 0.023350454996034387,
 'training_time_sec': 0.32570733900138293,
 'final_train_loss': 0.6531727313995361,
 'final_test_loss': 0.6796792149543762,
 'final_train_metric': 0.8666666666666667,
 'final_test_metric': 0.8,
 'optimizer': 'Adam',
 'seed': None,
 'epochs_completed': 30,
 'early_stopped': False,
 'early_stop_reason': None,
 'zero_acceptance_patience': 6,
 'final_zero_acceptance_streak': 0}

## Optimization using LBFGS-style second-order optimizer

In [5]:
loss_fn = nn.CrossEntropyLoss()
lbfgs_model = QuadraticMLP(4, [4], 3)
classical_device = "cpu" 
lbfgs_optimizer = torch.optim.LBFGS(lbfgs_model.parameters(), 
                              lr=0.01,
                              max_iter=1,
                              history_size=100,
                              line_search_fn=None,
                              )

In [6]:
train(
    model=lbfgs_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=lbfgs_optimizer,
    c_device=classical_device,
    epochs=30,
    experiment_name = EXPERIMENT_NAME,
    run_name="lbfgs-optimizer"
)

2026/05/01 12:08:51 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 000 | train_loss=1.2069 | train_acc=0.457 | test_loss=1.2242 | test_acc=0.444 | 
Epoch 005 | train_loss=0.8882 | train_acc=0.514 | test_loss=0.9320 | test_acc=0.467 | 
Epoch 010 | train_loss=0.7847 | train_acc=0.857 | test_loss=0.8367 | test_acc=0.800 | 
Epoch 015 | train_loss=0.6984 | train_acc=0.857 | test_loss=0.7573 | test_acc=0.778 | 
Epoch 020 | train_loss=0.6369 | train_acc=0.867 | test_loss=0.7018 | test_acc=0.778 | 
Epoch 025 | train_loss=0.5824 | train_acc=0.876 | test_loss=0.6534 | test_acc=0.778 | 
Epoch 029 | train_loss=0.5408 | train_acc=0.867 | test_loss=0.6167 | test_acc=0.800 | 


{'run_id': '05d0062ce7314aa6a8d5b8b4767db6a2',
 'experiment_id': '831960980176491418',
 'experiment_name': 'optimizer-comparison-mlp-iris-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'optimization_time_sec': 0.018393464997643605,
 'sum_epoch_optimization_time_sec': 0.018393464997643605,
 'train_evaluation_time_sec': 0.014454801997999311,
 'test_evaluation_time_sec': 0.008898552994651254,
 'evaluation_time_sec': 0.023353354992650566,
 'training_time_sec': 0.14087836900034745,
 'final_train_loss': 0.5407841801643372,
 'final_test_loss': 0.6167276501655579,
 'final_train_metric': 0.8666666666666667,
 'final_test_metric': 0.8,
 'optimizer': 'LBFGS',
 'seed': None,
 'epochs_completed': 30,
 'early_stopped': False,
 'early_stop_reason': None,
 'zero_acceptance_patience': 6,
 'final_zero_acceptance_streak': 0}

## Optimization using Newton-Rapson method

In [7]:
loss_fn = nn.CrossEntropyLoss() 
newton_model = QuadraticMLP(4, [4], 3)
classical_device = "cpu"
newton_optimizer = NewtonOptimizer(newton_model.parameters(), 
                                   lr=1, 
                                   max_iter=1,
                                   damping=1e-4,
                                   )

In [8]:
train(
    model=newton_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=newton_optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS,
    experiment_name = EXPERIMENT_NAME,
    run_name="newton-optimizer"
)

Epoch 000 | train_loss=1.0977 | train_acc=0.410 | test_loss=1.1169 | test_acc=0.333 | 
Epoch 005 | train_loss=79.8505 | train_acc=0.333 | test_loss=79.8180 | test_acc=0.333 | 
Epoch 010 | train_loss=6187.5679 | train_acc=0.333 | test_loss=6065.2725 | test_acc=0.333 | 
Epoch 015 | train_loss=2045.6643 | train_acc=0.590 | test_loss=2022.4121 | test_acc=0.578 | 
Epoch 020 | train_loss=2212.7378 | train_acc=0.686 | test_loss=1859.4988 | test_acc=0.733 | 
Epoch 025 | train_loss=2499.5354 | train_acc=0.619 | test_loss=1632.9191 | test_acc=0.644 | 


2026/05/01 12:08:54 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 029 | train_loss=2901.8792 | train_acc=0.514 | test_loss=3239.1499 | test_acc=0.378 | 


{'run_id': 'ba346ba94c7e4789a373ce5bd5b65870',
 'experiment_id': '831960980176491418',
 'experiment_name': 'optimizer-comparison-mlp-iris-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'optimization_time_sec': 0.37423788900559884,
 'sum_epoch_optimization_time_sec': 0.37423788900559884,
 'train_evaluation_time_sec': 0.017793840999729582,
 'test_evaluation_time_sec': 0.010281172000759398,
 'evaluation_time_sec': 0.02807501300048898,
 'training_time_sec': 0.5184411530008219,
 'final_train_loss': 2901.879150390625,
 'final_test_loss': 3239.14990234375,
 'final_train_metric': 0.5142857142857142,
 'final_test_metric': 0.37777777777777777,
 'optimizer': 'NewtonOptimizer',
 'seed': None,
 'epochs_completed': 30,
 'early_stopped': False,
 'early_stop_reason': None,
 'zero_acceptance_patience': 6,
 'final_zero_acceptance_streak': 0}

## Optimization using Quadratic Annealer (cpu + momentum)

In [7]:
loss_fn = nn.CrossEntropyLoss()
model = QuadraticMLP(4, [4], 3)
classical_device = "cpu" 

sampler = build_sampler(mode="simulated")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=12,
    step_size=0.05,
    num_reads=100,
    beta=0.9,
)

In [8]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-optimizer"
)

Epoch 000 | train_loss=0.9669 | train_acc=0.476 | test_loss=1.0249 | test_acc=0.422 | 
Epoch 005 | train_loss=0.4107 | train_acc=0.876 | test_loss=0.4916 | test_acc=0.800 | 
Epoch 010 | train_loss=0.1958 | train_acc=0.952 | test_loss=0.3045 | test_acc=0.889 | 
Epoch 015 | train_loss=0.0660 | train_acc=0.971 | test_loss=0.1321 | test_acc=0.956 | 


2026/05/01 12:30:06 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 020 | train_loss=0.0571 | train_acc=0.981 | test_loss=0.1209 | test_acc=0.956 | 


{'run_id': '6781140e7dfc48618f27cb21f239160c',
 'experiment_id': '831960980176491418',
 'experiment_name': 'optimizer-comparison-mlp-iris-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'optimization_time_sec': 0.3882589430104417,
 'sum_epoch_optimization_time_sec': 0.3882589430104417,
 'train_evaluation_time_sec': 0.014330036003229907,
 'test_evaluation_time_sec': 0.0076970740028627915,
 'evaluation_time_sec': 0.0220271100060927,
 'training_time_sec': 0.5577047509996191,
 'final_train_loss': 0.05709623545408249,
 'final_test_loss': 0.12085086852312088,
 'final_train_metric': 0.9809523809523809,
 'final_test_metric': 0.9555555555555556,
 'optimizer': 'QuadraticAnnealingOptimizer',
 'seed': None,
 'epochs_completed': 23,
 'early_stopped': True,
 'early_stop_reason': 'acceptance_rate_was_zero_for_6_consecutive_epochs',
 'zero_acceptance_patience': 6,
 'final_zero_acceptance_streak': 6}

## Optimization using Quadratic Annealer (D-Wave + momentum)

In [11]:
loss_fn = nn.CrossEntropyLoss()
model = QuadraticMLP(4, [4], 3)
classical_device = "cpu" 

sampler = build_sampler(mode="dwave")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=12,
    step_size=0.05,
    num_reads=100,
    beta=0.9,
)

In [ ]:
# train(
#     model=model,
#     train_loader=train_loader,
#     test_loader=test_loader,
#     loss_fn=loss_fn,
#     optimizer=optimizer,
#     c_device=classical_device,
#     epochs=NUM_EPOCHS,
#     experiment_name = EXPERIMENT_NAME,
#     run_name="quadratic-annealer-dwave-optimizer"
# )

Epoch 000 | train_loss=1.1942 | train_acc=0.324 | test_loss=1.1992 | test_acc=0.333 | 
Epoch 005 | train_loss=0.5088 | train_acc=0.857 | test_loss=0.5539 | test_acc=0.778 | 
Epoch 010 | train_loss=0.2043 | train_acc=0.924 | test_loss=0.3310 | test_acc=0.867 | 
Epoch 015 | train_loss=0.0535 | train_acc=0.971 | test_loss=0.1813 | test_acc=0.956 | 


2026/04/27 21:06:40 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


{'run_id': '7025ae46259c4c31b826324c4aa90d1d',
 'experiment_id': '831960980176491418',
 'experiment_name': 'optimizer-comparison-mlp-iris-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'optimization_time_sec': 20.39436264098913,
 'sum_epoch_optimization_time_sec': 20.39436264098913,
 'train_evaluation_time_sec': 0.026294642986613326,
 'test_evaluation_time_sec': 0.009131466002145316,
 'evaluation_time_sec': 0.03542610898875864,
 'training_time_sec': 20.636611406000156,
 'final_train_loss': 0.04677055776119232,
 'final_test_loss': 0.13710738718509674,
 'final_train_metric': 0.9809523809523809,
 'final_test_metric': 0.9555555555555556,
 'optimizer': 'QuadraticAnnealingOptimizer',
 'seed': None,
 'epochs_completed': 20,
 'early_stopped': True,
 'early_stop_reason': 'acceptance_rate_was_zero_for_3_consecutive_epochs',
 'zero_acceptance_patience': 3,
 'final_zero_acceptance_streak': 3}